# Asteroid Family Identification via Hierarchical Clustering Method (HCM)

## 1. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import time
from scipy.cluster.hierarchy import linkage, fcluster
import pickle

# Zone boundaries from Table I, Zappalà et al. 1990
ZONE_BOUNDARIES = {
    2: (2.065, 2.3),
    3: (2.3,   2.501),
    4: (2.501, 2.825),
    5: (2.825, 2.958),
    6: (2.958, 3.030),
    7: (3.030, 3.278),
}

ZONE_CUTOFFS = {
    2: 60,
    3: 50,
    4: 45,
    5: 50,
    6: 40,
    7: 45,
}
# list of large families to test on
FAMILY_NAMES = {
    4: 'Vesta', 
    10: 'Hygiea',
    15: 'Eunomia', 
    20: 'Massalia', 
    24: 'Themis',
    31: 'Euphrosyne', 
    93: 'Minerva', 
    135: 'Hertha',
    158: 'Koronis', 
    163: 'Erigone',
    170: 'Maria', 
    221: 'Eos', 
    302: 'Clarissa', 
    375: 'Ursula', 
    480: 'Hansa',
    490: 'Veritas', 
    569: 'Misa', 
    668: 'Dora', 
    808: 'Merxia',
    847: 'Agnia', 
    1040: 'Natasha', 
    3: 'Juno',
    25: 'Phocaea',
    110: 'Lydia',
    145: 'Adeona',
    194: 'Prokne',
    283: 'Emma',
    293: 'Brasilia',
    396: 'Aeolia',
    606: 'Brangäne',
    778: 'Theobalda',
    945: 'Barcelona',
    1547: 'Nele',
    1658: 'Innes',
    1726: 'Hoffmeister',
    2076: 'Levin',
    3330: 'Gantrisch',
    3815: 'König',
    3827: 'Zdeněkhorský',
    10955: 'Harig'
}


## 2. Load Data

In [ ]:
# proper_asteroid_data.csv  : proper orbital elements (a, e, sin_i, ...)
# family_membership.csv     : AstDys family assignments (name, family1, ...)
proper   = pd.read_csv('input_data/proper_asteroid_data_clean.csv', low_memory=False)
families = pd.read_csv('input_data/family_membership.csv', low_memory=False)

df = proper.merge(families[['name', 'family1']], on='name', how='left')
df['family1'] = df['family1'].fillna(0).astype(int)

print(f"Total asteroids loaded: {len(df):,}")
print(f"Asteroids with family assignment: {(df['family1'] > 0).sum():,}")
print()

# Count members per target family
print("Target family sizes in dataset:")
for fid, fname in FAMILY_NAMES.items():
    n = (df['family1'] == fid).sum()
    print(f"  {fname:12s} (id={fid:4d}): {n:,}")

## 3. Visualise Known Families

In [ ]:
def plot_family(family_id, df=df, family_names=FAMILY_NAMES):
    """
    Plot a known family in all three proper-element planes:
    (a, e), (a, sin_i), (e, sin_i).
    Red = family members, grey = background.
    """
    name    = family_names.get(family_id, f'Family {family_id}')
    members = df[df['family1'] == family_id]
    bg      = df[df['family1'] != family_id]
    print(f"{name}: {len(members):,} members in dataset")

    fig, axes = plt.subplots(1, 3, figsize=(10, 5))
    for ax, (xcol, ycol) in zip(axes, [('a','e'), ('a','sin_i'), ('e','sin_i')]):
        ax.scatter(bg[xcol],      bg[ycol],      s=1, c='lightgray', alpha=0.3, label='background')
        ax.scatter(members[xcol], members[ycol], s=8, c='red',       alpha=0.7,
                   label=f'{name} (n={len(members):,})')
        ax.set_xlabel(xcol); ax.set_ylabel(ycol)
        ax.set_title(f'{name} — {xcol} vs {ycol}')
        ax.set_xlim(members[xcol].min()-0.1, members[xcol].max()+0.1)
        ax.set_ylim(members[ycol].min()-0.1, members[ycol].max()+0.1)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

plot_family(396)      # Aeolia

## 4. HCM Core Functions

### Distance metric (Zappalà et al. 1990, Eq. 2)

$$\delta v = n a' \sqrt{k_1 \left(\frac{\delta a'}{a'}\right)^2 + k_2 (\delta e')^2 + k_3 (\delta \sin i')^2}$$

with $k_1 = 5/4$, $k_2 = 2$, $k_3 = 2$ (standard metric coefficients, Section II of the paper).

$na'$ is the circular orbital velocity at $a'$. Units: AU/yr × 4740.9 = m/s.

In [ ]:
# Standard Metric coefficients — Zappalà et al. 1990, Section II
K1, K2, K3 = 5/4, 2, 2

AU_YR_TO_MS = 4740.9


def compute_distances(X, chunk_size=1000, verbose=True):
    """
    Compute all N*(N-1)/2 pairwise delta_v distances (Eq. 2) in row chunks
    to avoid allocating an N^2 array all at once.

    For each row i, we vectorise over all j > i simultaneously using numpy,
    so the inner loop is a numpy operation, not a Python loop per pair.
    Peak memory ~ chunk_size * N * 4 bytes (float32).

    Parameters
    ----------
    X          : array (N, 3) — columns [a', e', sin_i']
    chunk_size : rows processed per outer iteration (1000 → ~few hundred MB)

    Returns
    -------
    dist : condensed distance array, length N*(N-1)/2, dtype float32
           Compatible with scipy.cluster.hierarchy.linkage.
    """
    N       = len(X)
    n_pairs = N * (N - 1) // 2
    dist    = np.empty(n_pairs, dtype=np.float32)

    a, e, si = X[:, 0], X[:, 1], X[:, 2]
    pair_idx = 0

    for i_start in range(0, N, chunk_size):
        i_end = min(i_start + chunk_size, N)

        if verbose and i_start % (chunk_size * 20) == 0:
            print(f"    {100*pair_idx/n_pairs:.1f}%  "
                  f"({pair_idx:,} / {n_pairs:,} pairs)", flush=True)

        for i in range(i_start, i_end):
            j = i + 1
            if j >= N:
                continue

            # All pairs (i, j), (i, j+1), ..., (i, N-1) at once
            a0  = (a[i] + a[j:]) / 2.0
            n0  = 2.0 * np.pi / (a0 ** 1.5)   # Kepler: n = 2π / T = 2π / a^1.5
            da  = a[i] - a[j:]
            de  = e[i] - e[j:]
            dsi = si[i] - si[j:]

            dv = n0 * a0 * np.sqrt(
                K1 * (da / a0)**2 +
                K2 * de**2 +
                K3 * dsi**2
            ) * AU_YR_TO_MS

            n_new = len(dv)
            dist[pair_idx : pair_idx + n_new] = dv
            pair_idx += n_new

    return dist


def run_hcm_zone(df, zone, save=True, chunk_size=1000):
    """
    Run HCM on one zone of the asteroid belt.

    Steps:
      1. Filter df to the zone's semimajor axis range.
      2. Compute all pairwise delta_v distances.
      3. Build single-linkage dendrogram (scipy 'single' method).
         Single linkage means: distance(group A, group B) = min pairwise distance.
         This is exactly the update rule in step (3) of the paper.
      4. Cut dendrogram at ZONE_CUTOFFS[zone] to produce family labels.
      5. Return all groups with >= min_members members.

    Saves Z_zone{zone}.npy and zone{zone}_df.csv so you can re-cut
    at different thresholds later without recomputing (use recut_zone).

    Returns
    -------
    families : list of lists of asteroid name strings
    zone_df  : DataFrame for this zone with 'hcm_label' column added
    """
    a_min, a_max  = ZONE_BOUNDARIES[zone]
    cutoff        = ZONE_CUTOFFS[zone]

    zone_df = df[
        (df['a'] >= a_min) & (df['a'] < a_max)
    ].dropna(subset=['a', 'e', 'sin_i']).copy()

    n       = len(zone_df)
    n_pairs = n * (n - 1) // 2
    mem_mb  = n_pairs * 4 / 1e6
    print(f"Zone {zone} ({a_min}–{a_max} AU): {n:,} asteroids | "
          f"{n_pairs:,} pairs | ~{mem_mb:.0f} MB")

    X = zone_df[['a', 'e', 'sin_i']].values

    print(f"  Computing pairwise distances...")
    dist = compute_distances(X, chunk_size=chunk_size)

    print(f"  Building dendrogram (single linkage)...")
    Z = linkage(dist, method='single')

    if save:
        np.save(f'Z_zone{zone}.npy', Z)
        zone_df.to_csv(f'zone{zone}_df.csv', index=False)
        print(f"  Saved Z_zone{zone}.npy + zone{zone}_df.csv")

    labels  = fcluster(Z, t=cutoff, criterion='distance')
    zone_df = zone_df.copy()
    zone_df['hcm_label'] = labels

    families = _labels_to_families(zone_df)
    print(f"  → {len(families)} families at cutoff={cutoff} m/s")
    return families, zone_df


def recut_zone(zone, cutoff, min_members=5):
    """
    Re-cut a previously saved dendrogram at a new cutoff — runs instantly.
    Use this to experiment with cutoffs without recomputing distances.
    """
    Z       = np.load(f'output_data/Z_zone{zone}.npy')
    zone_df = pd.read_csv(f'output_data/zone{zone}_df.csv', low_memory=False)
    zone_df['name'] = zone_df['name'].astype(str).str.strip()
    labels  = fcluster(Z, t=cutoff, criterion='distance')
    zone_df['hcm_label'] = labels
    families = _labels_to_families(zone_df, min_members)
    largest  = len(families[0]) if families else 0
    print(f"Zone {zone} @ {cutoff} m/s → {len(families)} families, largest={largest:,}")
    return families, zone_df


def _labels_to_families(zone_df, min_members=5):
    """Convert hcm_label column to sorted list of member-name lists."""
    families = []
    for lbl in np.unique(zone_df['hcm_label']):
        members = zone_df[zone_df['hcm_label'] == lbl]['name'].tolist()
        if len(members) >= min_members:
            families.append(members)
    families.sort(key=len, reverse=True)
    return families


def run_all_zones(df, min_members=5):
    """Run HCM across all zones and return all families."""
    all_families = []k_means.py
    total_start  = time.time()

    for zone in ZONE_BOUNDARIES:
        t0   = time.time()
        fams, _ = run_hcm_zone(df, zone)
        all_families.extend(fams)
        print(f"  Zone {zone} done in {time.time()-t0:.1f}s\n")

    print(f"All zones done in {time.time()-total_start:.1f}s")
    print(f"Total: {len(all_families):,} families, "
          f"{sum(len(f) for f in all_families):,} asteroids assigned")
    return all_families


def recut_all_zones(cutoffs=ZONE_CUTOFFS, min_members=5):
    """
    Re-cut all saved dendrograms using the given cutoff dict.
    Runs in seconds — no distance recomputation needed.
    """
    all_families = []
    for zone, cutoff in cutoffs.items():
        fams, _ = recut_zone(zone, cutoff, min_members)
        all_families.extend(fams)
    print(f"\nTotal: {len(all_families):,} families, "
          f"{sum(len(f) for f in all_families):,} asteroids assigned")
    return all_families

## 5. Run HCM

Calls files which are calculated in compute_dendrograms.py

In [ ]:
# Check whether saved Z files exist for all zones
zones_missing = [z for z in ZONE_BOUNDARIES if not os.path.exists(f'output_data/Z_zone{z}.npy')]

if zones_missing:
    print(f"No saved dendrograms found for zones {zones_missing}.")
    print("Running full HCM — this may take several minutes...\n")
    all_families = run_all_zones(df)
else:
    print("Saved dendrograms found for all zones. Re-cutting from disk...\n")
    all_families = recut_all_zones()    

## 6. Evaluate Against AstDys Ground Truth

In [ ]:
def evaluate(all_families, df, family_names=FAMILY_NAMES, threshold=0.95):
    """
    Compare HCM output against AstDys ground truth.

    For each known family, find the HCM cluster that contains the most
    true members (voting). Report:
      - Completeness (recall)  = recovered / true_size
      - Precision              = recovered / found_size
      - F1                     = harmonic mean of completeness and precision

    A family 'passes' if completeness >= threshold.
    """
    # Map each asteroid name to the index of its HCM family
    name_to_fi = {}
    for i, fam in enumerate(all_families):
        for name in fam:
            name_to_fi[name] = i

    rows = []
    for fam_id, fam_name in family_names.items():
        true_members = set(df[df['family1'] == fam_id]['name'].tolist())

        vote = {}
        for name in true_members:
            if name in name_to_fi:
                fi = name_to_fi[name]
                vote[fi] = vote.get(fi, 0) + 1

        if not vote:
            completeness = precision = f1 = 0.0
            recovered = found_size = 0
        else:
            best_fi      = max(vote, key=vote.get)
            recovered    = vote[best_fi]
            found_size   = len(all_families[best_fi])
            completeness = recovered / len(true_members)
            precision    = recovered / found_size
            f1           = (2 * precision * completeness /
                            (precision + completeness)
                            if (precision + completeness) > 0 else 0.0)

        rows.append({
            'family':       fam_name,
            'true_size':    len(true_members),
            'recovered':    recovered,
            'found_size':   found_size,
            'completeness': round(completeness * 100, 1),
            'precision':    round(precision    * 100, 1),
            'f1':           round(f1           * 100, 1),
            'pass':         '✓' if completeness >= threshold else '✗',
        })

    results = pd.DataFrame(rows).sort_values('completeness', ascending=False)

    print(f"  {'Family':12} {'True':>6} {'Recov':>6} {'Found':>7} "
          f"{'Compl':>7} {'Prec':>7} {'F1':>7}  Pass")
    print("-" * 70)
    for _, r in results.iterrows():
        print(f"  {r['family']:12} {r['true_size']:>6,} {r['recovered']:>6,} "
              f"{r['found_size']:>7,} {r['completeness']:>6.1f}% "
              f"{r['precision']:>6.1f}% {r['f1']:>6.1f}%  {r['pass']}")

    n_pass = results['pass'].eq('✓').sum()
    print(f"\n{n_pass}/{len(results)} families reach {int(threshold*100)}% completeness")
    return results

print("Baseline: single-pass HCM before halo attachment, very very unsuccessful")
results = evaluate(all_families, df)

## 7. Visualise HCM Results vs Ground Truth

In [ ]:
def attach_to_cores(all_families, df, zone_qrl_cutoffs, zone_boundaries):
    K1, K2, K3 = 5/4, 2, 2
    AU_YR_TO_MS = 4740.9
    
    core_names = set(str(n).strip() for fam in all_families for n in fam)
    unassigned = df[~df['name'].isin(core_names)].dropna(subset=['a','e','sin_i']).copy()
    
    # Pre-calculate zones for all unassigned asteroids
    a_un_all = unassigned['a'].values
    unassigned_zones = np.full(len(a_un_all), -1)
    for zone, (a_min, a_max) in zone_boundaries.items():
        mask = (a_un_all >= a_min) & (a_un_all < a_max)
        unassigned_zones[mask] = zone
        
    final_families = [list(fam) for fam in all_families]
    df_indexed = df.set_index('name')
    
    # Step 1: Figure out which zone each family belongs to
    fam_zones = []
    valid_fam_members = []
    for fam in all_families:
        fam_str_names = [str(n).strip() for n in fam]
        valid_members = [n for n in fam_str_names if n in df_indexed.index]
        valid_fam_members.append(valid_members)
        
        if not valid_members:
            fam_zones.append(-1)
            continue
            
        members = df_indexed.loc[valid_members]
        core_a_mean = members['a'].mean()
        cz = -1
        for z, (a_min, a_max) in zone_boundaries.items():
            if a_min <= core_a_mean < a_max:
                cz = z
                break
        fam_zones.append(cz)

    # Step 2: Process zone-by-zone so local families compete fairly (Fast + Accurate)
    for current_zone in zone_boundaries.keys():
        if current_zone not in zone_qrl_cutoffs:
            continue
            
        qrl_cutoff = zone_qrl_cutoffs[current_zone]
        
        # Grab only the unassigned asteroids in THIS zone
        zone_mask = (unassigned_zones == current_zone)
        if not np.any(zone_mask):
            continue
            
        local_unassigned = unassigned[zone_mask]
        a_u = local_unassigned['a'].values
        e_u = local_unassigned['e'].values
        si_u = local_unassigned['sin_i'].values
        names_u = local_unassigned['name'].values
        
        # Arrays to track the absolute closest family for each asteroid in this zone
        best_dist = np.full(len(names_u), np.inf)
        best_fam  = np.full(len(names_u), -1, dtype=int)
        
        # Let all families IN THIS ZONE compute distances
        for fam_idx, cz in enumerate(fam_zones):
            if cz != current_zone:
                continue
                
            valid_members = valid_fam_members[fam_idx]
            members = df_indexed.loc[valid_members]
            a_m = members['a'].values
            e_m = members['e'].values
            si_m = members['sin_i'].values
            
            chunk = 500
            for i in range(0, len(names_u), chunk):
                a_chunk  = a_u[i:i+chunk, None]   
                e_chunk  = e_u[i:i+chunk, None]
                si_chunk = si_u[i:i+chunk, None]
                
                a0 = (a_chunk + a_m[None, :]) / 2.0          
                n0 = 2.0 * np.pi / (a0 ** 1.5)
                dv = n0 * a0 * np.sqrt(
                    K1 * ((a_chunk  - a_m[None,:])  / a0)**2 +
                    K2 * (e_chunk  - e_m[None,:])**2 +
                    K3 * (si_chunk - si_m[None,:])**2
                ) * AU_YR_TO_MS
                
                min_dv_to_fam = dv.min(axis=1) 
                
                # Update if this family is closer than previously checked families
                mask = min_dv_to_fam < best_dist[i:i+chunk]
                best_dist[i:i+chunk][mask] = min_dv_to_fam[mask]
                best_fam[i:i+chunk][mask]  = fam_idx
                
        # Step 3: Assign asteroids to their absolute closest family (if under QRL)
        for j, name in enumerate(names_u):
            if best_dist[j] <= qrl_cutoff and best_fam[j] != -1:
                final_families[best_fam[j]].append(str(name))
                
    return final_families

In [ ]:
# Force type consistency on the main dataframe
df['name'] = df['name'].astype(str).str.strip()

# 1. Get the high-precision cores (no chaining!)
# These are the tight cutoffs we discovered from your sweep tables
core_families = recut_all_zones(cutoffs=ZONE_CUTOFFS)

# 2. Define the Zone-Specific Quasi-Random Levels (QRLs)
QRL_CUTOFFS = {2:70, 3:90, 4:100, 5:120, 6:60, 7:65}


# 3. Append the halos using nearest-member distance (One step outward)
print("Attaching halos to core families...")
if os.path.exists('output_data/final_families.pkl'):
    with open('output_data/final_families.pkl', 'rb') as f:
        final_families = pickle.load(f)
    print("Loaded final_families from disk")
else:
    print("No saved families found — run attach_to_cores first")
    final_families = attach_to_cores(
        all_families=core_families, 
        df=df, 
        zone_qrl_cutoffs=QRL_CUTOFFS, 
        zone_boundaries=ZONE_BOUNDARIES
    )

# 4. Evaluate the modernized Two-Step HCM!
results = evaluate(final_families, df, family_names=FAMILY_NAMES)

In [ ]:
final_families_save = final_families

In [ ]:
def plot_family_comparison(family_name, target_id, all_hcm_families, df):
    """
    Plots the HCM discovered family vs the AstDys Ground Truth.
    """
    # 1. Get the Ground Truth members
    truth_mask = df['family1'] == target_id
    truth_df = df[truth_mask]
    
    # 2. Get the HCM members
    # Find the cluster that has the majority of the target family's members
    best_fam = []
    max_overlap = 0
    truth_names = set(truth_df['name'].astype(str).str.strip())
    
    for fam in all_hcm_families:
        fam_set = set([str(n).strip() for n in fam])
        overlap = len(fam_set.intersection(truth_names))
        if overlap > max_overlap:
            max_overlap = overlap
            best_fam = fam
            
    hcm_df = df[df['name'].isin(best_fam)]
    
    if len(hcm_df) == 0:
        print(f"Could not find an HCM cluster for {family_name}.")
        return

    # 3. Create the plots
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Define bounds with a little padding
    a_min, a_max = truth_df['a'].min() - 0.05, truth_df['a'].max() + 0.05
    e_min, e_max = truth_df['e'].min() - 0.02, truth_df['e'].max() + 0.02
    si_min, si_max = truth_df['sin_i'].min() - 0.02, truth_df['sin_i'].max() + 0.02
    
    # Background subset for faster rendering
    bg_df = df[(df['a'] > a_min) & (df['a'] < a_max)]

    # --- Plot 1: a vs e ---
    ax = axes[0]
    ax.scatter(bg_df['a'], bg_df['e'], s=1, c='lightgray', alpha=0.5, zorder=1)
    ax.scatter(truth_df['a'], truth_df['e'], s=8, c='blue', alpha=0.6, label='AstDys Truth', zorder=2)
    ax.scatter(hcm_df['a'], hcm_df['e'], s=2, c='red', alpha=0.8, label='Your HCM', zorder=3)
    ax.set_xlim(a_min, a_max)
    ax.set_ylim(e_min, e_max)
    ax.set_xlabel('Proper Semi-major Axis (a) [AU]')
    ax.set_ylabel('Proper Eccentricity (e)')
    ax.legend()
    
    # --- Plot 2: a vs sin(i) ---
    ax = axes[1]
    ax.scatter(bg_df['a'], bg_df['sin_i'], s=1, c='lightgray', alpha=0.5, zorder=1)
    ax.scatter(truth_df['a'], truth_df['sin_i'], s=8, c='blue', alpha=0.6, label='AstDys Truth', zorder=2)
    ax.scatter(hcm_df['a'], hcm_df['sin_i'], s=2, c='red', alpha=0.8, label='Your HCM', zorder=3)
    ax.set_xlim(a_min, a_max)
    ax.set_ylim(si_min, si_max)
    ax.set_xlabel('Proper Semi-major Axis (a) [AU]')
    ax.set_ylabel('Proper Sine of Inclination (sin i)')
    ax.legend()

    plt.suptitle(f'Asteroid Family: {family_name}', fontsize=16)
    plt.tight_layout()
    plt.show()
plot_family_comparison('Aeolia', 396, final_families, df) #very good
plot_family_comparison('Koronis', 158, final_families, df) #very good
plot_family_comparison('Clarissa', 302, final_families, df) #high precision but mid completion
plot_family_comparison('phocaea', 25, final_families, df) #full precision but low completion 